In [3]:
import os
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from openpyxl import Workbook
from openpyxl.styles import Alignment
from openpyxl.utils.dataframe import dataframe_to_rows



# =========================
# Chrome配置
# =========================

options = Options()

options.add_argument("--start-maximized")

# 不保存登录状态
options.add_argument("--incognito")


options.add_experimental_option(
    "excludeSwitches",
    ["enable-automation"]
)


options.add_argument(
    "--disable-blink-features=AutomationControlled"
)



webdriver_path = r"C:\WebDriver\chromedriver.exe"



driver = webdriver.Chrome(
    service=Service(webdriver_path),
    options=options
)


print("Chrome启动成功")



# =========================
# 打开视频号后台
# =========================


driver.get(
    "https://channels.weixin.qq.com/platform/post/list"
)


print("请扫码登录")



while True:

    result = driver.execute_script(
        """
        return document.querySelector('wujie-app') != null;
        """
    )


    if result:
        break


    time.sleep(1)



print("登录成功")

time.sleep(2)



# =========================
# 提取当前页数据
# =========================


def extract_video_data():

    print("开始提取当前页")


    script = """

    let app=document.querySelector('wujie-app');


    if(!app){
        return [];
    }


    let root=app.shadowRoot;



    let items=root.querySelectorAll(
        'div.post-feed-item'
    );



    let result=[];



    items.forEach(item=>{


        let title =
        item.querySelector(
            '.post-title'
        )?.innerText || "";



        let date =
        item.querySelector(
            '.time-label'
        )?.innerText || "";



        let stats =
        item.querySelectorAll(
            '.post-data .data-item'
        );



        function getValue(index){


            if(!stats[index]){
                return "0";
            }


            let text =
            stats[index].innerText;



            let arr =
            text.split("\\n");



            return arr[arr.length-1] || "0";

        }



        result.push({

            "标题":title,

            "日期":
            date.split(" ")[0],


            "播放量":
            getValue(0),


            "点赞":
            getValue(1),


            "评论":
            getValue(2),


            "转发":
            getValue(3),


            "收藏":
            getValue(4)

        });



    });



    return result;


    """



    data = driver.execute_script(script)


    print(
        "当前页数量:",
        len(data)
    )


    return data

# =========================
# 点击下一页
# =========================


def next_page():


    script = """

    let app=document.querySelector('wujie-app');


    if(!app){

        return false;

    }



    let root=app.shadowRoot;



    if(!root){

        return false;

    }




    let buttons =
    root.querySelectorAll(
        '.weui-desktop-pagination__nav a'
    );



    for(let btn of buttons){


        if(
            btn.innerText.includes("下一页")
        ){


            // 检查是否禁用

            if(
                btn.className.includes("disabled")
            ){

                return false;

            }



            btn.click();


            return true;


        }


    }



    return false;


    """



    result = driver.execute_script(
        script
    )



    print(
        "下一页点击:",
        result
    )


    return result





# =========================
# 数字转换
# =========================


def convert_to_number(value):


    if not value:

        return 0



    value=str(value).strip()



    if "万" in value:


        try:


            return int(

                float(
                    value.replace("万","")
                )
                *
                10000

            )


        except:


            return 0




    try:


        return int(

            value.replace(",","")

        )


    except:


        return 0
    
    # =========================
# 保存Excel
# =========================


def save_to_excel(data):


    if not data:

        print("没有数据")

        return



    df=pd.DataFrame(data)



    # 固定Excel列顺序

    df=df[
        [
            "标题",
            "日期",
            "播放量",
            "点赞",
            "评论",
            "转发",
            "收藏"
        ]
    ]



    # 数字转换

    for col in [

        "播放量",
        "点赞",
        "评论",
        "转发",
        "收藏"

    ]:


        df[col]=df[col].apply(
            convert_to_number
        )



    file_path=os.path.join(

        os.path.expanduser("~"),

        "Desktop",

        "微信视频信息.xlsx"

    )



    wb=Workbook()


    ws=wb.active


    ws.title="视频信息"



    for row in dataframe_to_rows(

        df,

        index=False,

        header=True

    ):

        ws.append(row)




    # 设置列宽

    ws.column_dimensions["A"].width=45

    ws.column_dimensions["B"].width=20



    for col in [

        "C",
        "D",
        "E",
        "F",
        "G"

    ]:

        ws.column_dimensions[col].width=12





    # 数字右对齐

    for row in ws.iter_rows(

        min_row=2,

        min_col=3,

        max_col=7

    ):

        for cell in row:

            cell.alignment=Alignment(
                horizontal="right"
            )



    # 标题居中

    for cell in ws[1]:

        cell.alignment=Alignment(
            horizontal="center"
        )



    wb.save(file_path)



    print(
        "Excel保存成功:",
        file_path
    )


    os.startfile(file_path)





# =========================
# 主程序
# 自动翻页直到结束
# =========================


if __name__=="__main__":


    all_data=[]


    page=1



    while True:



        print(
            "===================="
        )


        print(
            "正在抓取第",
            page,
            "页"
        )



        time.sleep(0.5)



        current_data=extract_video_data()



        if not current_data:


            print(
                "当前页没有数据，结束"
            )

            break




        all_data.extend(
            current_data
        )



        print(
            "当前累计作品:",
            len(all_data)
        )



        # 点击下一页

        has_next=next_page()



        if not has_next:


            print(
                "没有下一页，抓取结束"
            )


            break



        page+=1



        # 等待下一页加载

        time.sleep(0.5)





    print(
        "全部完成，总作品:",
        len(all_data)
    )



    save_to_excel(all_data)



    driver.quit()

Chrome启动成功
请扫码登录
登录成功
正在抓取第 1 页
开始提取当前页
当前页数量: 20
当前累计作品: 20
下一页点击: True
正在抓取第 2 页
开始提取当前页
当前页数量: 20
当前累计作品: 40
下一页点击: True
正在抓取第 3 页
开始提取当前页
当前页数量: 20
当前累计作品: 60
下一页点击: True
正在抓取第 4 页
开始提取当前页
当前页数量: 20
当前累计作品: 80
下一页点击: True
正在抓取第 5 页
开始提取当前页
当前页数量: 20
当前累计作品: 100
下一页点击: True
正在抓取第 6 页
开始提取当前页
当前页数量: 20
当前累计作品: 120
下一页点击: True
正在抓取第 7 页
开始提取当前页
当前页数量: 20
当前累计作品: 140
下一页点击: True
正在抓取第 8 页
开始提取当前页
当前页数量: 20
当前累计作品: 160
下一页点击: True
正在抓取第 9 页
开始提取当前页
当前页数量: 20
当前累计作品: 180
下一页点击: True
正在抓取第 10 页
开始提取当前页
当前页数量: 20
当前累计作品: 200
下一页点击: True
正在抓取第 11 页
开始提取当前页
当前页数量: 7
当前累计作品: 207
下一页点击: False
没有下一页，抓取结束
全部完成，总作品: 207
Excel保存成功: C:\Users\qieziclub1660ti\Desktop\微信视频信息.xlsx
